# Part1: manual adding the cookie and get the data for the course you choose
# iif selenium fails, try this part 

In [5]:
!pip install browser-cookie3 requests

In [3]:
import requests
import json
import pandas as pd
import os

In [1]:


def fetch_and_save_reviews(course_name, cookie_str, output_root="./ust_reviews"):
    # 1. Setup URL and Parameters( the request library make the url look like this https://ust.space/review/COMP1021/get?single=false&composer=false&preferences%5Bsort%5D=0&preferences%5BfilterInstructor%5D=0&preferences%5BfilterSemester%5D=0&preferences%5BfilterRating%5D=0) )
    url = f"https://ust.space/review/{course_name}/get"
    params = {
        "single": "false",
        "composer": "false",
        "preferences[sort]": 0,
        "preferences[filterInstructor]": 0,
        "preferences[filterSemester]": 0,
        "preferences[filterRating]": 0
    }
    
    # 2. Setup Headers (Simulating browser behavior)( simulate the human browser header, especially the cookie and user agent)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
        "X-Requested-With": "XMLHttpRequest",
        "Cookie": cookie_str
    }

    # 3. Send Request
    print(f"Fetching data for {course_name}...")
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() # Raise an exception for HTTP errors
        data = response.json()
        reviews = data.get('reviews', [])
        print(f"Successfully retrieved {len(reviews)} reviews!")
    except Exception as e:
        print(f"Failed to fetch data: {e}")
        return

    if not reviews:
        print("No review data found. Process stopped.")
        return

    # 4. Prepare Output Directory: ust_reviews/<course_name>
    output_folder = os.path.join(output_root, course_name)
    os.makedirs(output_folder, exist_ok=True)
    print(f"Output folder: {output_folder}")

    json_path = os.path.join(output_folder, f"{course_name}_reviews.json")
    csv_path = os.path.join(output_folder, f"{course_name}_reviews.csv")

    # 5. Save as JSON
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(reviews, f, ensure_ascii=False, indent=4)
    print(f"JSON saved to: {json_path}")

    # 6. Save as CSV (encoding good for chinese comment and excel )
    df = pd.DataFrame(reviews)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"CSV saved to: {csv_path}")



In [4]:
# --- Main Entry Point ---
if __name__ == "__main__":
    # Configuration

# Target URL (the API endpoint that returns the reviews in JSON format) 
# #- you can find this by inspecting the network requests in your browser's developer tools when you load the reviews page( check the network and its the fetch/XHR, cfind the one with get method )
# usually the largest one, eg for COMP1021 one it is 600+ mb
#  url eg: https://ust.space/review/MATH1013/get?single=false&composer=false&preferences%5Bsort%5D=0&preferences%5BfilterInstructor%5D=0&preferences%5BfilterSemester%5D=0&preferences%5BfilterRating%5D=0
# copy the cookie from the request header, and paste it here as a string
    
    MY_COOKIE = "XSRF-TOKEN=eyJpdiI6ImtrYVZGZlZmdzRKYXhEQkZlZ1pDQ3c9PSIsInZhbHVlIjoiN0crRUFGMnI0T2ZTUWE0c0RVemlxZ2tVdWVEWFJGV2ZWYlJaUExWL1ZuWXdVaC9zcWJuRDNLdkVEZ1M4TTloMDRlTit2T09RQ2R4ZDhNOG9lajhHWUpuTVlSOFJzRUJLQXFDTS9xb3RWTit6VktOOVBNaUM1SGhCUmNLQjFCalMiLCJtYWMiOiJmZGFhYjA4NjQ4OTQ3Y2I5NTJiMDFjOTVlNGM4Y2E1ODkyNGY4NTg0N2YyOGJmNGViNDRiMGY3MjY3NWMyMjAyIiwidGFnIjoiIn0%3D; ustspace_session=eyJpdiI6IkFxVTJWenlXMUpQNVkydU5nVkFrbkE9PSIsInZhbHVlIjoiZXZmakI3RUR3SjVpUENPc05HdTQ4M0srNy9jRC9LQ2kyaXkvazY5R3ZPbzBJTDkwTlZjanAvSDdoMEFlY0diTkp0RGU5d2xCZCtqK0Z1RkJIN25yQmtzSVlMcnNDVVM3QU1DdXJobmQyYnplYUhDRktOVWZCaEFuYmMwQmIvTmMiLCJtYWMiOiI0ZWZiMGYxZWI0ZTJjMmY4NGIwM2NmODhhNmU4MWNjMmZhNmNjZGRmNjc5MjAwYTI2OWE4M2E0NjA1OTMwMDg3IiwidGFnIjoiIn0%3D"

    TARGET_COURSES = ["MATH1013"]  # change target courses here
    SAVE_ROOT = "./ust_reviews"
    
    for course in TARGET_COURSES:
        fetch_and_save_reviews(course, MY_COOKIE, SAVE_ROOT)

Fetching data for MATH1013...
Successfully retrieved 1079 reviews!
Output folder: ./ust_reviews\MATH1013
JSON saved to: ./ust_reviews\MATH1013\MATH1013_reviews.json
CSV saved to: ./ust_reviews\MATH1013\MATH1013_reviews.csv


# Part2: using selenium to get the cookie and run the script 

In [6]:
pip install selenium


  Obtaining dependency information for selenium from https://files.pythonhosted.org/packages/a8/d6/e4160989ef6b272779af6f3e5c43c3ba9be6687bdc21c68c3fb220e555b3/selenium-4.41.0-py3-none-any.whl.metadata
  Using cached selenium-4.41.0-py3-none-any.whl.metadata (7.5 kB)
  Obtaining dependency information for certifi>=2026.1.4 from https://files.pythonhosted.org/packages/9a/3c/c17fb3ca2d9c3acff52e30b309f538586f9f5b9c9cf454f3845fc9af4881/certifi-2026.2.25-py3-none-any.whl.metadata
  Obtaining dependency information for trio<1.0,>=0.31.0 from https://files.pythonhosted.org/packages/1c/93/dab25dc87ac48da0fe0f6419e07d0bfd98799bed4e05e7b9e0f85a1a4b4b/trio-0.33.0-py3-none-any.whl.metadata
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Obtaining dependency information for trio-websocket<1.0,>=0.12.2 from https://files.pythonhosted.org/packages/c7/19/eb640a397bba49ba49ef9dbe2e7e5c04202ba045b6ce2ec36e9cadc51e04/trio_websocket-0.12.2-py3-none-any.whl.metadata
  Using cached trio_we

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 2.1.1 requires sentencepiece, which is not installed.
botocore 1.27.59 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
httpcore 1.0.5 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.


In [20]:
pip install --upgrade typing_extensions

In [7]:
import os
import time
import json
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


In [8]:
# ==========================================
# 1. Automated Login & Cookie Extraction Module
# ==========================================
def _click_first_available(driver, wait, locators, step_name):
    """Try multiple locators and click the first matched clickable element."""
    for by, value in locators:
        try:
            elem = wait.until(EC.element_to_be_clickable((by, value)))
            driver.execute_script("arguments[0].click();", elem)
            print(f"[OK] {step_name}")
            return True
        except Exception:
            continue
    print(f"[WARN] {step_name} not found, continue...")
    return False


def _switch_to_latest_window(driver):
    """If login opens a new tab/window, switch to it."""
    handles = driver.window_handles
    if len(handles) > 1:
        driver.switch_to.window(handles[-1])


def _find_element_any_frame(driver, wait, locator):
    """Find an element in default content or any iframe."""
    driver.switch_to.default_content()
    try:
        return wait.until(EC.presence_of_element_located(locator))
    except Exception:
        pass

    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    for frame in iframes:
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame(frame)
            return wait.until(EC.presence_of_element_located(locator))
        except Exception:
            continue

    driver.switch_to.default_content()
    raise Exception(f"Element not found in default content or iframes: {locator}")


def _type_first_available(driver, wait, locators, value, field_name):
    """Type value into the first matched input field."""
    for locator in locators:
        try:
            elem = _find_element_any_frame(driver, wait, locator)
            elem.clear()
            elem.send_keys(value)
            print(f"[OK] Filled {field_name}")
            return True
        except Exception:
            continue
    raise Exception(f"Cannot find {field_name} input")


def get_auth_cookie(username, password, login_url="https://ust.space"):
    print("Launching browser for guided login flow...")

    options = webdriver.ChromeOptions()
    options.add_argument('--log-level=3')
    # Keep browser visible for login steps.

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 25)
    cookie_str = ""

    try:
        driver.get(login_url)

        # Step 1: click "Join Us Now"
        _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//a[contains(., 'Join Us Now')]|//button[contains(., 'Join Us Now')]|//*[contains(., 'Join Us Now') and (self::a or self::button)]"),
                (By.XPATH, "//a[contains(., 'Join Us')]|//button[contains(., 'Join Us')]")
            ],
            "Clicked Join Us Now"
        )

        # Step 2: click "Return to login page"
        _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//a[contains(., 'Return to login page')]|//button[contains(., 'Return to login page')]|//*[contains(., 'Return to login page') and (self::a or self::button)]"),
                (By.XPATH, "//a[contains(., 'Return to log in')]|//button[contains(., 'Return to log in')]"),
                (By.XPATH, "//a[contains(., 'Log in')]|//button[contains(., 'Log in')]|//a[contains(., 'Sign in')]|//button[contains(., 'Sign in')]")
            ],
            "Clicked Return to login page"
        )

        # Login may open in a new tab/window
        time.sleep(1)
        _switch_to_latest_window(driver)

        # Step 3: fill USTSPACE login form
        _type_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//input[contains(@placeholder, 'ITSC Network Account Name')]"),
                (By.XPATH, "//input[contains(@name, 'username') or contains(@id, 'username') or contains(@name, 'account')]"),
                (By.XPATH, "//input[@type='text']")
            ],
            username,
            "username"
        )

        _type_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//input[contains(@placeholder, 'Password')]"),
                (By.XPATH, "//input[@type='password']"),
                (By.XPATH, "//input[contains(@name, 'password') or contains(@id, 'password')]")
            ],
            password,
            "password"
        )

        driver.switch_to.default_content()

        # Submit login form
        clicked_login = _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//button[contains(., 'Login')]|//a[contains(., 'Login')]"),
                (By.XPATH, "//input[@type='submit']")
            ],
            "Clicked Login"
        )
        if not clicked_login:
            # Fallback: press Enter in password field
            try:
                pass_input = _find_element_any_frame(driver, wait, (By.XPATH, "//input[@type='password']"))
                pass_input.send_keys(Keys.RETURN)
                print("[OK] Submitted form with ENTER")
            except Exception:
                pass

        # Return focus to latest window after auth redirects
        time.sleep(1)
        _switch_to_latest_window(driver)

        # Step 4: click "Course Review"
        _click_first_available(
            driver,
            wait,
            [
                (By.XPATH, "//a[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'course review')]|//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'course review')]"),
                (By.XPATH, "//a[contains(@href, '/review')]|//button[contains(@onclick, 'review')]")
            ],
            "Clicked Course Review"
        )

        # Step 5: get cookies after navigation to review area
        print("Waiting for authentication tokens...")
        for _ in range(30):
            cookies = {c['name']: c['value'] for c in driver.get_cookies()}
            if 'ustspace_session' in cookies and 'XSRF-TOKEN' in cookies:
                cookie_str = "; ".join([f"{k}={v}" for k, v in cookies.items()])
                print("-> Authentication successful. Cookie acquired.")
                break
            time.sleep(1)

    except Exception as e:
        print(f"Failed to acquire cookie: {e}")

    finally:
        driver.quit()

    return cookie_str

# --- Example Usage ---
# my_cookie = get_auth_cookie("your_account", "your_password")
# print(my_cookie)

In [9]:

# ==========================================
# 2. Scraper Module
# ==========================================
def fetch_and_save_reviews(course_name, cookie_str, output_root="./ust_reviews"):
    url = f"https://ust.space/review/{course_name}/get"
    params = {
        "single": "false", "composer": "false",
        "preferences[sort]": 0, "preferences[filterInstructor]": 0,
        "preferences[filterSemester]": 0, "preferences[filterRating]": 0
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json",
        "X-Requested-With": "XMLHttpRequest",
        "Cookie": cookie_str
    }

    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() 
        reviews = response.json().get('reviews', [])
        
        if not reviews:
            print(f"[{course_name}] No reviews found. Skipping.")
            return

        # Ensure directory exists: ust_reviews/<course_name>
        output_folder = os.path.join(output_root, course_name)
        os.makedirs(output_folder, exist_ok=True)
        print(f"Output folder: {output_folder}")

        json_path = os.path.join(output_folder, f"{course_name}_reviews.json")
        csv_path = os.path.join(output_folder, f"{course_name}_reviews.csv")

        # Save JSON & CSV
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(reviews, f, ensure_ascii=False, indent=4)
        
        pd.DataFrame(reviews).to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"[{course_name}] Saved {len(reviews)} reviews.")

    except Exception as e:
        print(f"[{course_name}] Fetch failed: {e}")


In [10]:

# ==========================================
# 3. Main Execution
# ==========================================
if __name__ == "__main__":
    # WARNING: Do not upload your credentials to public repositories (e.g., GitHub)
    MY_ACCOUNT = "thnganaa" 
    MY_PASSWORD = "aazh0994"
    
    SAVE_ROOT = "./ust_reviews"
    TARGET_COURSES = ["COMP1021", "MATH1014"]
    
    # 1. Authenticate
    auth_cookie = get_auth_cookie(MY_ACCOUNT, MY_PASSWORD)
    
    # 2. Scrape Data
    if auth_cookie:
        print("\nStarting data extraction...")
        for course in TARGET_COURSES:
            fetch_and_save_reviews(course, auth_cookie, SAVE_ROOT)
            time.sleep(2) # Polite delay between server requests
            
        print("\nAll tasks completed successfully!")
    else:
        print("\nProgram terminated due to missing authentication cookie.")

Launching browser for guided login flow...
[OK] Clicked Join Us Now
[OK] Clicked Return to login page
[OK] Filled username
[OK] Filled password
[OK] Clicked Login
[OK] Clicked Course Review
Waiting for authentication tokens...
-> Authentication successful. Cookie acquired.

Starting data extraction...
Output folder: ./ust_reviews\COMP1021
[COMP1021] Saved 1234 reviews.
Output folder: ./ust_reviews\MATH1014
[MATH1014] Saved 842 reviews.

All tasks completed successfully!
